# Telecom Customer Churn – Model Training
Load data, explore, train a Random Forest pipeline, evaluate, save.

In [ ]:
import pandas as pd, numpy as np, json, joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay

## 1. Load data

In [ ]:
df = pd.read_csv('telecom_customers.csv')
print(df.shape)
df.head()

## 2. Explore

In [ ]:
print(df.churn.value_counts(normalize=True))
df.groupby('contract').churn.mean().plot(kind='bar', title='Churn rate by contract')
plt.show()

## 3. Preprocessing and model

In [ ]:
X = df.drop(columns=['customer_id','churn']); y = df.churn
cat = ['gender','contract','internet_service','payment_method']
num = [c for c in X.columns if c not in cat]
pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat), ('n', StandardScaler(), num)])
model = Pipeline([('pre', pre), ('rf', RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, class_weight='balanced'))])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model.fit(Xtr, ytr)

## 4. Evaluate

In [ ]:
pred = model.predict(Xte); proba = model.predict_proba(Xte)[:,1]
print(classification_report(yte, pred))
print('ROC-AUC:', roc_auc_score(yte, proba))
ConfusionMatrixDisplay(confusion_matrix(yte, pred)).plot()
plt.show()

## 5. Save model and metrics

In [ ]:
joblib.dump(model, 'churn_model.joblib')
json.dump({'roc_auc': float(roc_auc_score(yte, proba))}, open('metrics.json','w'), indent=2)